# Prédiction de Toxicité avec Keras et RDKit
Dans ce notebook, nous allons construire un réseau de neurones avec Keras (TensorFlow) pour classifier des molécules (Toxique vs Non-Toxique) en utilisant les empreintes digitales de Morgan (Morgan Fingerprints).

In [ ]:
# Installation des dépendances (décommentez et exécutez si nécessaire)
# !pip install rdkit tensorflow scikit-learn numpy

In [ ]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

# 1. DONNÉES D'EXEMPLE
# 1 = Toxique, 0 = Non toxique
data = [
    ("c1ccccc1", 1),       # Benzène
    ("CCO", 0),            # Éthanol
    ("C(=O)O", 0),         # Acide formique
    ("C1=CC=C(C=C1)O", 1), # Phénol
    ("C", 0),              # Méthane
    ("C1=CC=C(C=C1)Cl", 1) # Chlorobenzène
]

smiles_list = [item[0] for item in data]
labels = [item[1] for item in data]

In [ ]:
# 2. CONVERSION EN VECTEURS (Morgan Fingerprints)
def smiles_to_fingerprint(smiles, radius=2, n_bits=2048):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return np.zeros((n_bits,))
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, radius, nBits=n_bits)
    return np.array(fp)

X = np.array([smiles_to_fingerprint(s) for s in smiles_list])
y = np.array(labels)

# 3. SÉPARATION ENTRAÎNEMENT / TEST
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)

print(f"Taille des données d'entraînement : {X_train.shape}")
print(f"Taille des données de test : {X_test.shape}")

In [ ]:
# 4. CRÉATION DU MODÈLE KERAS
model = keras.Sequential([
    # Couche d'entrée qui correspond à la taille de notre fingerprint (2048)
    keras.Input(shape=(2048,)),
    
    # Première couche cachée avec 64 neurones et activation ReLU
    layers.Dense(64, activation='relu'),
    
    # Couche de régularisation (Dropout) pour éviter le surapprentissage
    layers.Dropout(0.2),
    
    # Deuxième couche cachée avec 32 neurones
    layers.Dense(32, activation='relu'),
    
    # Couche de sortie : 1 neurone avec activation Sigmoid pour une probabilité (0 à 1)
    layers.Dense(1, activation='sigmoid')
])

# Compilation du modèle
# On utilise binary_crossentropy car c'est un problème de classification binaire
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

model.summary()

In [ ]:
# 5. ENTRAÎNEMENT DU MODÈLE
print("Début de l'entraînement...")
history = model.fit(
    X_train, y_train,
    epochs=50,           # Nombre de passages sur l'ensemble des données
    batch_size=2,        # Mise à jour des poids toutes les 2 molécules
    verbose=1,
    validation_data=(X_test, y_test)
)
print("Entraînement terminé !")

In [ ]:
# 6. ÉVALUATION ET PRÉDICTION SUR UNE NOUVELLE MOLÉCULE
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Précision sur le jeu de test : {accuracy * 100:.2f}%\n")

nouvelle_molecule_smiles = "c1ccccc1C" # Toluène
nouvelle_empreinte = smiles_to_fingerprint(nouvelle_molecule_smiles)
nouvelle_empreinte = nouvelle_empreinte.reshape(1, -1) # Keras attend (batch_size, n_features)

# Prédire la probabilité (entre 0 et 1)
probabilite = model.predict(nouvelle_empreinte, verbose=0)[0][0]

print(f"--- Analyse du Toluène ({nouvelle_molecule_smiles}) ---")
if probabilite >= 0.5:
    print(f"Prédiction : TOXIQUE (Confiance : {probabilite*100:.2f}%)")
else:
    print(f"Prédiction : NON TOXIQUE (Confiance : {(1-probabilite)*100:.2f}%)")